### 1) Install all dependencies

In [ ]:
!pip install --upgrade transformers accelerate datasets sentencepiece sacrebleu torch
!pip install sacrebleu
!pip install evaluate
!pip install torchmetrics

### 2) Load the model and the test data

In [ ]:
import transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import pandas as pd

model_name = "../models/____MODEL____NAME____"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, dtype="auto", attn_implementation="sdpa")


df = pd.read_csv('../data/test/dev.de-hsb.de', sep='\t', header = None, on_bad_lines='skip')
df_test = pd.read_csv('../data/test/dev.de-hsb.hsb', sep='\t', header = None, on_bad_lines='skip')

src_sentences = df[0].tolist()
src_sentences

### 3) Move the model to the gpu

In [ ]:
model = model.to("cuda")      

### 3) Perform the translation

In [ ]:
tokenizer.src_lang = "de_Latn"
tokenizer.tgt_lang = "hsb_Latn"

translated = []
batch_size = 64


for i in range(0, len(df), batch_size):
    batch = src_sentences[i:i+batch_size]

    # Tokenize whole batch
    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True
      ).to(model.device)

    # Translate whole batch
    outputs = model.generate(
          **inputs,
          forced_bos_token_id=tokenizer.convert_tokens_to_ids("hsb_Latn"),
          max_length=128
    )

      # Decode whole batch
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    translated.extend(decoded)
translated

### 4) Prepare the reference sentences

In [ ]:
test_refs = df_test[0].tolist()
references = [[ref] for ref in test_refs]
references

### 5) Calculate Bleu Score

In [ ]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(translated, references)
print(f"BLEU score: {bleu.score:.2f}")

### 6) Calculate chrF++ score

In [ ]:
score = sacrebleu.corpus_chrf(translated, references, word_order=2)
print("chrF++ (corpus):", score.score)